# How to: Visual Autoencoders (VAEs)

Welcome to Generative Deep Learning. In this tutorial, we will build a **Variational Autoencoder (VAE)** to learn the visual distribution of handwritten digits and generate entirely new, fake digits from pure mathematics.

### The Analytical Architecture:
1. **The Encoder:** Takes an input image $X$ and maps it to two vectors: a mean vector $\mu$ and a log-variance vector $\log(\sigma^2)$.
2. **The Reparameterization Trick:** We cannot mathematically perform backpropagation through a random sampling process. To solve this, we sample a random noise vector $\epsilon$ from a standard normal distribution, and calculate our latent vector $z$ as:
   $$z = \mu + \sigma \odot \epsilon$$
3. **The Decoder:** Takes the sampled latent vector $z$ and maps it back into a reconstructed image $\hat{X}$.

Let's begin by setting up our PyTorch environment.

In [ ]:
# Cell 2: Environment Setup and Imports
# In a real environment, you need: !pip install torch torchvision matplotlib

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# Set random seed for analytical reproducibility
torch.manual_seed(42)

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Step 1: Ingesting the Visual Data

To train our visual generative model, we will use the classic **MNIST** dataset (28x28 pixel grayscale images of handwritten digits). 

We will load the data in batches. Because we are using fully connected linear layers for this basic VAE (rather than Convolutions, which are used for advanced VAEs), we will eventually flatten these 2D images into 1D arrays of 784 pixels.

In [ ]:
# Cell 4: Dataset Loading

batch_size = 128

# Transformations: Convert images to PyTorch tensors (values between 0.0 and 1.0)
transform = transforms.ToTensor()

# Download and load the training data
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# Create a DataLoader to handle batching and shuffling
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

print(f"Dataset loaded. Total training images: {len(train_dataset)}")
print(f"Image tensor shape per batch: {next(iter(train_loader))[0].shape}") 
# Expected: [128, 1, 28, 28] (Batch, Channels, Height, Width)

## Step 2: Designing the VAE Architecture

This is the mathematical core of the tutorial. We define the `VAE` class.

**Key Analytical Focus:**
Notice the `reparameterize` function. If we simply generated $z$ randomly from the distribution $\mathcal{N}(\mu, \sigma^2)$, the calculus chain rule would break, and the network couldn't learn. By shifting the randomness to an external variable $\epsilon$, the network can backpropagate gradients perfectly through $\mu$ and $\sigma$.

In [ ]:
# Cell 6: The VAE Neural Network

class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20):
        super(VAE, self).__init__()
        
        # --- ENCODER ---
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        # The encoder outputs TWO parallel vectors: Mean and Log-Variance
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        # --- DECODER ---
        self.fc3 = nn.Linear(latent_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        """Maps an image to latent distribution parameters."""
        h1 = F.relu(self.fc1(x))
        return self.fc_mu(h1), self.fc_logvar(h1)

    def reparameterize(self, mu, logvar):
        """The Reparameterization Trick: z = mu + std * epsilon"""
        std = torch.exp(0.5 * logvar) # Convert log variance to standard deviation
        eps = torch.randn_like(std)   # Sample epsilon from standard normal
        return mu + eps * std

    def decode(self, z):
        """Maps a latent vector back to an image."""
        h3 = F.relu(self.fc3(z))
        # We use Sigmoid here because our pixel values must be between 0 and 1
        return torch.sigmoid(self.fc4(h3))

    def forward(self, x):
        """The complete pipeline."""
        mu, logvar = self.encode(x.view(-1, 784)) # Flatten image
        z = self.reparameterize(mu, logvar)
        reconstructed_x = self.decode(z)
        return reconstructed_x, mu, logvar

# Instantiate the model
latent_dimensions = 20
model = VAE(latent_dim=latent_dimensions).to(device)
print(model)

## Step 3: The Loss Function (ELBO)

Training a VAE requires minimizing the **Evidence Lower Bound (ELBO)**. Analytically, this consists of two competing mathematical forces:

1. **Reconstruction Loss (BCE/MSE):** Measures how accurately the decoded image matches the original input image. (Force 1: Make the image look good).
2. **Kullback-Leibler (KL) Divergence:** Measures how far our learned latent distributions deviate from a standard Normal distribution $\mathcal{N}(0, 1)$. (Force 2: Force the distributions to overlap near the origin, ensuring the latent space is continuous and smooth, with no "dead zones").

$$\text{Loss} = \text{Reconstruction Error} + D_{KL}(q(z|x) || p(z))$$

In [ ]:
# Cell 8: Loss Function and Optimizer

def vae_loss_function(reconstructed_x, x, mu, logvar):
    """Calculates the combined Reconstruction and KL Divergence loss."""
    
    # 1. Reconstruction Loss (Binary Cross Entropy)
    # Compares flattened input to flattened output
    BCE = F.binary_cross_entropy(reconstructed_x, x.view(-1, 784), reduction='sum')

    # 2. KL Divergence
    # Analytical formula for KL divergence between two Gaussians
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    # Total loss
    return BCE + KLD

# Define the optimizer (Adam is standard for generative models)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
print("Loss function and optimizer defined.")

## Step 4: The Training Loop

We will now train the network. For every image in the batch:
1. Feed it forward to get the reconstruction, the means, and the log-variances.
2. Calculate the combined ELBO loss.
3. Backpropagate the gradients to update the weights.

*Note: For the sake of this tutorial's speed, we will only train for 5 epochs. In a real scenario, you would train for 50+ epochs.*

In [ ]:
# Cell 10: Training the Model

epochs = 5
model.train() # Set model to training mode

print("--- Starting Training ---")

for epoch in range(epochs):
    train_loss = 0
    for batch_idx, (data, _) in enumerate(train_loader):
        # Move data to GPU if available
        data = data.to(device)
        
        # 1. Zero the gradients
        optimizer.zero_grad()
        
        # 2. Forward pass
        reconstructed_batch, mu, logvar = model(data)
        
        # 3. Calculate Loss
        loss = vae_loss_function(reconstructed_batch, data, mu, logvar)
        
        # 4. Backward pass and optimize
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
        
    # Calculate average loss per image
    avg_loss = train_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1:2d} | Average ELBO Loss: {avg_loss:.4f}")

print("--- Training Complete ---")

## Step 5: Generative Inference (The Magic)

Because we forced the latent space to approximate a standard Normal distribution $\mathcal{N}(0, 1)$ using KL Divergence, the network knows how to decode *any* random coordinate near the origin into a valid image.

We will bypass the Encoder entirely! We will generate pure mathematical noise, pass it straight into the Decoder, and generate brand new handwritten digits that have never existed before.

In [ ]:
# Cell 12: Generating New Visuals

model.eval() # Set model to evaluation mode

# 1. Sample 8 random latent vectors from a standard normal distribution
# Shape: [8 images, 20 latent dimensions]
with torch.no_grad():
    random_latent_vectors = torch.randn(8, latent_dimensions).to(device)
    
    # 2. Decode the noise into images
    generated_images = model.decode(random_latent_vectors).cpu()

# 3. Visualize the generated images
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i in range(8):
    # Reshape the 1D 784-pixel array back into a 2D 28x28 visual grid
    img = generated_images[i].view(28, 28).numpy()
    axes[i].imshow(img, cmap='gray')
    axes[i].axis('off')

plt.suptitle("AI-Generated Digits from Pure Mathematical Noise", fontsize=16)
plt.show()

print("Analytical Conclusion: The VAE successfully learned the continuous visual manifold of handwritten digits.")
print("By simply feeding it random Gaussian coordinates, the Decoder reconstructs logical geometric shapes!")